<a href="https://colab.research.google.com/github/majdaliengmech-glitch/Quadruped-RL/blob/main/locomotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Locomotion in MuJoCo Playground (rewritten, with fixed budgets)

Trains four policies with **hardcoded step budgets** and prints a **time estimate before each run**:

| # | Run | Robot | Steps (hardcoded) | ~Episodes* | Est. time on A100 / RTX 4090 class |
|---|-----|-------|------------------:|-----------:|-----------------------------------:|
| 1 | `joystick` | Unitree Go1 | 200 M | ~200 k | ~8 min + JIT |
| 2 | `handstand` | Unitree Go1 | 100 M | ~100-200 k | ~8 min + JIT |
| 3 | `handstand_finetune` | Unitree Go1 (from #2) | 50 M | ~50-100 k | ~4 min + JIT |

\*Brax PPO counts **steps**, not episodes. `episodes ≈ steps / episode_length`; the exact number is printed at runtime.

**Whole notebook: ~35 min of training, ~50 min wall clock** including install, JIT compilation and rendering on an A100 / 4090-class GPU. Slower Colab GPUs (L4, T4) take several times longer, so set `QUICK_TEST = True` first.

**Requires a GPU runtime.**

## 1. Setup

In [ ]:
#@title Install (one command; versions pinned as in the original notebook)
!pip install -q -U "jax[cuda12]==0.10.2" flax optax brax mujoco mujoco_mjx playground "warp-lang>=1.11" mediapy
!command -v ffmpeg >/dev/null || (apt update -qq && apt install -y -qq ffmpeg)

In [ ]:
#@title GPU check + headless rendering (EGL) + XLA flag
import os
import subprocess

try:
  GPU_NAME = subprocess.check_output(
      ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
  ).strip().splitlines()[0]
except Exception as e:
  raise RuntimeError(
      "Cannot communicate with GPU. Use a GPU runtime "
      "(Runtime > Change runtime type)."
  ) from e
print("GPU:", GPU_NAME)

# Let glvnd find the NVIDIA EGL driver (missing on Colab by default).
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
  with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : { "library_path" : "libEGL_nvidia.so.0" }
}
""")

os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"

import mujoco
mujoco.MjModel.from_xml_string("<mujoco/>")
print("MuJoCo OK")

In [ ]:
#@title Imports
import functools
import json
from datetime import datetime

import jax
import numpy as np
from jax import numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
from IPython.display import clear_output, display

from brax.io import model
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from etils import epath
from flax.training import orbax_utils
from orbax import checkpoint as ocp

from mujoco_playground import registry, wrapper
from mujoco_playground.config import locomotion_params
from mujoco_playground._src.gait import draw_joystick_command

np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
#@title Compatibility patch for newer JAX
# Newer JAX removed jax.device_put_replicated, but Brax still calls it.
# Re-create it only if it is missing.
if not hasattr(jax, "device_put_replicated"):
  from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

  def _device_put_replicated(x, devices):
    devices = list(devices)
    mesh = Mesh(np.array(devices), ("i",))
    sharding = NamedSharding(mesh, P("i"))
    def put(a):
      a = jp.asarray(a)
      return jax.device_put(jp.stack([a] * len(devices)), sharding)
    return jax.tree_util.tree_map(put, x)

  jax.device_put_replicated = _device_put_replicated

## 2. Hardcoded training budget

Edit **only this cell** to change how long things train.

* `num_timesteps` = total environment steps (this replaces the library default).
* `num_evals` = how many times PPO evaluates, plots and **saves a checkpoint** (one every `num_timesteps / num_evals` steps).
* `STEPS_PER_SEC` = PPO throughput measured on a single **A100** in the MuJoCo Playground paper (Go1 joystick: 417k, handstand: 204k, Berkeley humanoid: 120k steps/s). It is used for the time estimate.
* `GPU_SLOWDOWN` is a **rough guess** of how much slower your GPU is than an A100 (not measured). Override it if you know better.

In [ ]:
QUICK_TEST = False      # True -> every budget below is divided by QUICK_DIVISOR (smoke test, a few minutes)
QUICK_DIVISOR = 20
SEED = 0
BASE_DIR = "runs"       # point this at a Google Drive folder to survive Colab disconnects

TRAIN = {
    "joystick": dict(
        env="Go1JoystickFlatTerrain", num_timesteps=200_000_000, num_evals=10),
    "handstand": dict(
        env="Go1Handstand", num_timesteps=100_000_000, num_evals=10),
    "handstand_finetune": dict(
        env="Go1Handstand", num_timesteps=50_000_000, num_evals=5),
    "humanoid": dict(
        env="BerkeleyHumanoidJoystickFlatTerrain", num_timesteps=100_000_000, num_evals=10),
}

# Fine-tuning rules for the handstand (smoother, lower power).
HANDSTAND_FINETUNE_OVERRIDES = dict(
    energy_termination_threshold=400,
    energy=-0.003,
    dof_acc=-2.5e-7,
)

# PPO steps/second on one A100 (MuJoCo Playground paper).
STEPS_PER_SEC = {
    "Go1JoystickFlatTerrain": 417_000,
    "Go1Handstand": 204_000,
    "BerkeleyHumanoidJoystickFlatTerrain": 120_000,
}
JIT_OVERHEAD_SEC = 120   # guess: one-time compile per training run

# Rough guess of slowdown vs. an A100 (NOT measured).
_SLOWDOWN_GUESS = {"H100": 0.7, "A100": 1.0, "4090": 1.0, "L40": 1.3, "A10": 2.0, "L4": 3.0, "T4": 5.0}
GPU_SLOWDOWN = next((v for k, v in _SLOWDOWN_GUESS.items() if k in GPU_NAME), 3.0)
print(f"GPU={GPU_NAME}  assumed slowdown vs A100: x{GPU_SLOWDOWN}")

RUN_DIR = epath.Path(BASE_DIR).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = {}

In [ ]:
#@title Estimate (printed before anything is trained)
def steps_for(name):
  n = TRAIN[name]["num_timesteps"]
  return n // QUICK_DIVISOR if QUICK_TEST else n


def estimate(name, verbose=True):
  spec = TRAIN[name]
  cfg = registry.get_default_config(spec["env"])
  steps = steps_for(name)
  sps = STEPS_PER_SEC[spec["env"]] / GPU_SLOWDOWN
  seconds = steps / sps + JIT_OVERHEAD_SEC
  episodes = steps / cfg.episode_length
  if verbose:
    print(f"{name:20s} {steps/1e6:6.0f}M steps | episode_length={cfg.episode_length} "
          f"| ~{episodes:,.0f} episodes | ~{seconds/60:4.1f} min")
  return seconds


total = sum(estimate(n) for n in TRAIN)
print(f"\nEstimated total training time: ~{total/60:.0f} min "
      f"(+ install/render, QUICK_TEST={QUICK_TEST})")

## 3. Reusable helpers

* `train(name, ...)` trains one run, **checkpoints every eval**, saves the **final params** and reward curve, and records timing in `RESULTS`.
* Each run has its own checkpoint folder (`runs/checkpoints/<name>`), so the handstand fine-tune no longer overwrites the first handstand run's checkpoints (which happened when both shared one folder).

In [ ]:
def latest_checkpoint(ckpt_dir):
  ckpts = [p for p in epath.Path(ckpt_dir).glob("*") if p.is_dir() and p.name.isdigit()]
  if not ckpts:
    raise FileNotFoundError(f"No checkpoints in {ckpt_dir}")
  return sorted(ckpts, key=lambda p: int(p.name))[-1]


def train(name, env_cfg=None, restore_from=None):
  spec = TRAIN[name]
  env_name = spec["env"]
  env_cfg = env_cfg if env_cfg is not None else registry.get_default_config(env_name)

  # Hardcoded budget replaces the library default.
  ppo_params = locomotion_params.brax_ppo_config(env_name)
  ppo_params["num_timesteps"] = steps_for(name)
  ppo_params["num_evals"] = spec["num_evals"]

  ckpt_dir = RUN_DIR / "checkpoints" / name
  ckpt_dir.mkdir(parents=True, exist_ok=True)
  with open(ckpt_dir / "env_config.json", "w") as f:
    json.dump(env_cfg.to_dict(), f, indent=2, default=str)
  with open(ckpt_dir / "ppo_params.json", "w") as f:
    json.dump(ppo_params.to_dict(), f, indent=2, default=str)

  estimate(name)
  hist = dict(step=[], reward=[], err=[], t=[])
  t_start = datetime.now()

  def save_ckpt(step, make_policy, params):
    del make_policy
    ocp.PyTreeCheckpointer().save(
        ckpt_dir / f"{step}", params, force=True,
        save_args=orbax_utils.save_args_from_target(params))

  def progress(step, metrics):
    hist["step"].append(step)
    hist["reward"].append(float(metrics["eval/episode_reward"]))
    hist["err"].append(float(metrics["eval/episode_reward_std"]))
    hist["t"].append(datetime.now())
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(hist["step"], hist["reward"], yerr=hist["err"], color="blue")
    ax.set_xlim([0, ppo_params["num_timesteps"] * 1.25])
    ax.set_xlabel("# environment steps")
    ax.set_ylabel("reward per episode")
    elapsed = (hist["t"][-1] - t_start).total_seconds() / 60
    ax.set_title(f"{name}: reward={hist['reward'][-1]:.2f}  ({elapsed:.1f} min elapsed)")
    display(fig)
    plt.close(fig)

  kwargs = dict(ppo_params)
  net_cfg = kwargs.pop("network_factory", None)
  network_factory = (
      functools.partial(ppo_networks.make_ppo_networks, **net_cfg)
      if net_cfg else ppo_networks.make_ppo_networks)

  make_inference_fn, params, metrics = ppo.train(
      **kwargs,
      network_factory=network_factory,
      randomization_fn=registry.get_domain_randomizer(env_name),
      progress_fn=progress,
      policy_params_fn=save_ckpt,
      environment=registry.load(env_name, config=env_cfg),
      eval_env=registry.load(env_name, config=env_cfg),
      wrap_env_fn=wrapper.wrap_for_brax_training,
      restore_checkpoint_path=restore_from,
      seed=SEED,
  )

  model.save_params(str(ckpt_dir / "final_params"), params)
  jit_s = (hist["t"][0] - t_start).total_seconds()
  train_s = (hist["t"][-1] - hist["t"][0]).total_seconds()
  RESULTS[name] = dict(
      steps=ppo_params["num_timesteps"],
      episodes=ppo_params["num_timesteps"] / env_cfg.episode_length,
      jit_min=jit_s / 60, train_min=train_s / 60,
      final_reward=hist["reward"][-1], ckpt_dir=str(ckpt_dir))
  print(f"[{name}] compile+first eval: {jit_s/60:.1f} min | training: {train_s/60:.1f} min | "
        f"final reward: {hist['reward'][-1]:.2f}")
  return dict(make_inference_fn=make_inference_fn, params=params, env_cfg=env_cfg,
              ckpt_dir=ckpt_dir, hist=hist)

In [ ]:
#@title Rollout / render / plot helpers
def make_eval(env_name, res, env_cfg=None):
  env_cfg = env_cfg if env_cfg is not None else registry.get_default_config(env_name)
  env = registry.load(env_name, config=env_cfg)
  return (env, jax.jit(env.reset), jax.jit(env.step),
          jax.jit(res["make_inference_fn"](res["params"], deterministic=True)))


def render_video(env, traj, mod_fns=None, camera="track", render_every=2,
                 width=640, height=480, show_pert=True, save_as=None):
  scene_option = mujoco.MjvOption()
  scene_option.geomgroup[2] = True
  scene_option.geomgroup[3] = False
  scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
  scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
  scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = show_pert
  fps = 1.0 / env.dt / render_every
  kw = {}
  if mod_fns is not None:
    kw["modify_scene_fns"] = mod_fns[::render_every]
  frames = env.render(traj[::render_every], camera=camera, scene_option=scene_option,
                      width=width, height=height, **kw)
  media.show_video(frames, fps=fps, loop=False)
  if save_as:
    media.write_video(str(RUN_DIR / f"{save_as}.mp4"), frames, fps=fps)
    print("saved", RUN_DIR / f"{save_as}.mp4")


def _smooth(x, k=10):
  return np.convolve(x, np.ones(k) / k, mode="same")


def plot_feet(swing_peak, max_foot_height):
  swing_peak = np.stack([np.asarray(s) for s in swing_peak])
  fig, axs = plt.subplots(2, 2)
  for i, (ax, nm, c) in enumerate(zip(axs.flat, ["FR", "FL", "RR", "RL"], "rgby")):
    ax.plot(swing_peak[:, i], color=c)
    ax.set_ylim([0, max_foot_height * 1.25])
    ax.axhline(max_foot_height, color="k", linestyle="--")
    ax.set_title(nm); ax.set_xlabel("time"); ax.set_ylabel("height")
  plt.tight_layout(); plt.show()


def plot_tracking(linvel, angvel, commands, cfg):
  lin = np.stack([np.asarray(v) for v in linvel])
  ang = np.stack([np.asarray(v) for v in angvel])
  cmd = np.stack([np.asarray(c) for c in commands])
  series = [lin[:, 0], lin[:, 1], ang[:, 2]]
  fig, axes = plt.subplots(3, 1, figsize=(10, 10))
  for i, (ax, s, lab) in enumerate(zip(axes, series, ["dx", "dy", "dyaw"])):
    ax.plot(_smooth(s), label="actual (10-step avg)")
    ax.plot(cmd[:, i], "r--", label="command")
    lim = cfg.command_config.a[i]
    ax.set_ylim(-lim, lim); ax.set_ylabel(lab)
  axes[0].legend()
  plt.show()

## 4. Quadruped: Go1 joystick

The policy tracks a commanded forward speed, sideways speed and turn rate `[dx, dy, dyaw]`.
Domain randomization (friction, armature, torso center of mass, link masses, ...) is applied automatically via the registry so the policy is robust enough for sim-to-real.

In [ ]:
joystick = train("joystick")

In [ ]:
#@title Joystick rollout (edit the command here)
x_vel = 0.0    #@param {type: "number"}
y_vel = 0.0    #@param {type: "number"}
yaw_vel = 3.14 #@param {type: "number"}


def rollout_joystick(res, command_fn, n_steps, kick=(0.0, 0.0),
                     kick_duration=(0.05, 0.2), seed=0):
  env_name = TRAIN["joystick"]["env"]
  cfg = registry.get_default_config(env_name)
  cfg.pert_config.enable = True
  cfg.pert_config.velocity_kick = [3.0, 6.0]
  cfg.pert_config.kick_wait_times = [5.0, 15.0]
  cfg.command_config.a = [1.5, 0.8, 2 * jp.pi]
  env, jit_reset, jit_step, jit_inf = make_eval(env_name, res, cfg)

  def maybe_push(rng, state):
    # kick=(0,0) disables pushes; set e.g. kick=(3.0, 6.0) to shove the robot.
    if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
      rng, k1, k2 = jax.random.split(rng, 3)
      mag = jax.random.uniform(k1, minval=kick[0], maxval=kick[1])
      dur_s = jax.random.uniform(k2, minval=kick_duration[0], maxval=kick_duration[1])
      state.info["pert_mag"] = mag
      state.info["pert_duration"] = jp.round(dur_s / env.dt).astype(jp.int32)
      state.info["pert_duration_seconds"] = dur_s
    return rng

  rng = jax.random.PRNGKey(seed)
  state = jit_reset(rng)
  rng = maybe_push(rng, state)
  out = dict(traj=[], mod_fns=[], swing=[], linvel=[], angvel=[], cmds=[], env=env, cfg=cfg)
  last_cmd = None
  for i in range(n_steps):
    cmd = command_fn(i)
    if last_cmd is None or not np.array_equal(np.asarray(cmd), np.asarray(last_cmd)):
      print(f"step {i}: command = {np.asarray(cmd)}")
      last_cmd = cmd
    state.info["command"] = cmd
    rng = maybe_push(rng, state)
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inf(state.obs, act_rng)
    state = jit_step(state, ctrl)
    state.info["command"] = cmd

    out["traj"].append(state)
    out["swing"].append(state.info["swing_peak"])
    out["linvel"].append(env.get_global_linvel(state.data))
    out["angvel"].append(env.get_gyro(state.data))
    out["cmds"].append(cmd)

    xyz = np.array(state.data.xpos[env._torso_body_id]) + np.array([0, 0, 0.2])
    x_axis = state.data.xmat[env._torso_body_id, 0]
    yaw = -np.arctan2(x_axis[1], x_axis[0])
    out["mod_fns"].append(functools.partial(
        draw_joystick_command, cmd=cmd, xyz=xyz, theta=yaw,
        scl=abs(cmd[0]) / cfg.command_config.a[0]))
  return out


const_cmd = jp.array([x_vel, y_vel, yaw_vel])
r = rollout_joystick(joystick, lambda i: const_cmd, n_steps=joystick["env_cfg"].episode_length)
render_video(r["env"], r["traj"], r["mod_fns"], save_as="joystick_constant")
plot_feet(r["swing"], r["cfg"].reward_config.max_foot_height)
plot_tracking(r["linvel"], r["angvel"], r["cmds"], r["cfg"])

In [ ]:
#@title Ramp test: forward command +0.25 m/s every 200 steps
ramp = lambda i: jp.array([0.25 * (i // 200), 0.0, 0.0])
r = rollout_joystick(joystick, ramp, n_steps=1400)
render_video(r["env"], r["traj"], r["mod_fns"], save_as="joystick_ramp")
plot_feet(r["swing"], r["cfg"].reward_config.max_foot_height)
plot_tracking(r["linvel"], r["angvel"], r["cmds"], r["cfg"])

## 5. Go1 handstand: train, then fine-tune from a checkpoint

Stage 1 trains with the default rewards (works, but jittery and high power). Stage 2 restarts from the **latest stage-1 checkpoint** with energy and joint-acceleration penalties for a smoother, lower-power policy. Fine-tuning uses its own, smaller budget.

In [ ]:
handstand = train("handstand")

In [ ]:
def rollout_handstand(name, res, env_cfg, n_episodes=10, seed=12345):
  env_name = TRAIN[name]["env"]
  env, jit_reset, jit_step, jit_inf = make_eval(env_name, res, env_cfg)
  rng = jax.random.PRNGKey(seed)
  traj, power_signed, power_abs, height = [], [], [], []
  for _ in range(n_episodes):
    rng, reset_rng = jax.random.split(rng)
    state = jit_reset(reset_rng)
    for _ in range(env_cfg.episode_length // 2):
      act_rng, rng = jax.random.split(rng)
      ctrl, _ = jit_inf(state.obs, act_rng)
      state = jit_step(state, ctrl)
      traj.append(state)
      frc = state.data.actuator_force
      qvel = state.data.qvel[6:]
      power_signed.append(jp.sum(frc * qvel))
      power_abs.append(jp.sum(jp.abs(frc * qvel)))
      height.append(state.data.qpos[2])
  render_video(env, traj, camera="side", show_pert=False, save_as=f"{name}")
  print(f"[{name}] max power (signed): {float(jp.max(jp.array(power_signed))):.1f} | "
        f"max |power|: {float(jp.max(jp.array(power_abs))):.1f} | "
        f"mean |power|: {float(jp.mean(jp.array(power_abs))):.1f}")
  return env


rollout_handstand("handstand", handstand, handstand["env_cfg"])

In [ ]:
#@title Fine-tune (restores latest stage-1 checkpoint)
ft_cfg = registry.get_default_config(TRAIN["handstand_finetune"]["env"])
ft_cfg.energy_termination_threshold = HANDSTAND_FINETUNE_OVERRIDES["energy_termination_threshold"]
ft_cfg.reward_config.energy = HANDSTAND_FINETUNE_OVERRIDES["energy"]
ft_cfg.reward_config.dof_acc = HANDSTAND_FINETUNE_OVERRIDES["dof_acc"]

restore_path = latest_checkpoint(handstand["ckpt_dir"])
print("Restoring from", restore_path)
handstand_ft = train("handstand_finetune", env_cfg=ft_cfg, restore_from=restore_path)

In [ ]:
rollout_handstand("handstand_finetune", handstand_ft, ft_cfg)

## 7. Summary: what each run actually cost

In [ ]:
print(f"{'run':20s} {'steps':>8s} {'episodes':>10s} {'compile':>9s} {'train':>8s} {'reward':>8s}")
for n, r_ in RESULTS.items():
  print(f"{n:20s} {r_['steps']/1e6:7.0f}M {r_['episodes']:10,.0f} "
        f"{r_['jit_min']:8.1f}m {r_['train_min']:7.1f}m {r_['final_reward']:8.2f}")
print(f"\nTotal: {sum(r_['jit_min'] + r_['train_min'] for r_ in RESULTS.values()):.1f} min")
print("Videos, checkpoints and final_params are in:", RUN_DIR)